# 02 - Data Cleaning and Standardization

## Core Question

How can the raw NFL datasets be transformed into consistent, reliable, and joinable data sources for feature engineering and modeling?

## Purpose

This notebook cleans and standardizes the raw datasets collected in `01_Data_Collection.ipynb`.

The goal is to create trustworthy intermediate datasets before any predictive features are engineered.

Key cleaning priorities include:

- Standardizing NFL team abbreviations across data sources and seasons
- Separating regular season and postseason records where appropriate
- Standardizing season and week fields
- Preserving stable player identifiers
- Identifying duplicate records
- Understanding missing data patterns
- Handling differences in historical data coverage
- Creating consistent data types
- Validating row counts and key relationships
- Saving cleaned datasets for downstream feature engineering

No predictive modeling or football ratings are created in this notebook.

# Load Raw Data

The raw datasets collected in Notebook 01 are loaded from disk without modification.

Keeping raw and cleaned data separate ensures that every cleaning step can be reproduced and audited.

In [2]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ROSTER_DIR = DATA_DIR / "roster"
SCHEDULE_DIR = DATA_DIR / "schedules"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

c:\Users\efriedman\Desktop\NFL-Season-Projections


In [3]:
SEASONS = list(range(2015, 2026))

schedules = pl.read_parquet(
    SCHEDULE_DIR / "nfl_schedules_2015_2025.parquet"
)

rosters = pl.read_parquet(
    ROSTER_DIR / "nfl_rosters_2015_2025.parquet"
)

player_stats = pl.read_parquet(
    RAW_DIR / "player_stats_2015_2025.parquet"
)

participation = pl.read_parquet(
    RAW_DIR / "participation_2016_2025.parquet"
)

snap_counts = pl.read_parquet(
    RAW_DIR / "snap_counts_2015_2025.parquet"
)

depth_charts = pl.read_parquet(
    RAW_DIR / "depth_charts_raw.parquet"
)

draft_picks = pl.read_parquet(
    RAW_DIR / "draft_picks_2015_2025.parquet"
)

injuries = pl.read_parquet(
    RAW_DIR / "injuries_2015_2025.parquet"
)

contracts = pl.read_parquet(
    RAW_DIR / "contracts_raw.parquet"
)

trades = pl.read_parquet(
    RAW_DIR / "trades_raw.parquet"
)

weekly_rosters = pl.read_parquet(
    RAW_DIR / "weekly_rosters_2015_2025.parquet"
)

print("All non-play-by-play raw datasets loaded successfully.")

All non-play-by-play raw datasets loaded successfully.


# Standardize Team Abbreviations

NFL datasets do not always use the same team abbreviations across sources or seasons.

A canonical abbreviation system is defined here so every downstream dataset uses one consistent team identifier.

In [4]:
TEAM_MAP = {
    # Arizona
    "ARZ": "ARI",

    # Baltimore
    "BLT": "BAL",

    # Cleveland
    "CLV": "CLE",

    # Houston
    "HST": "HOU",

    # Jacksonville
    "JAC": "JAX",

    # Green Bay
    "GNB": "GB",

    # Kansas City
    "KAN": "KC",

    # Los Angeles Rams / St. Louis Rams
    "STL": "LA",
    "SL": "LA",
    "LAR": "LA",

    # Los Angeles Chargers / San Diego Chargers
    "SD": "LAC",
    "SDG": "LAC",

    # Las Vegas / Oakland Raiders
    "OAK": "LV",
    "LVR": "LV",

    # New England
    "NWE": "NE",

    # New Orleans
    "NOR": "NO",

    # San Francisco
    "SFO": "SF",

    # Tampa Bay
    "TAM": "TB",

    # Washington
    "WSH": "WAS",
    "WFT": "WAS"
}

def standardize_team_column(df, column_name):
    if column_name not in df.columns:
        return df

    return df.with_columns(
        pl.col(column_name)
        .replace(TEAM_MAP)
        .alias(column_name)
    )

In [5]:
test_teams = pl.DataFrame({
    "team": ["GB", "JAC", "STL", "SD", "OAK", "WSH", "WFT"]
})

standardize_team_column(test_teams, "team")

team
str
"""GB"""
"""JAX"""
"""LA"""
"""LAC"""
"""LV"""
"""WAS"""
"""WAS"""


## Apply Team Standardization

The canonical team mapping is now applied across the raw datasets.

Only fields representing NFL team identities are standardized. Original raw files remain unchanged on disk.

In [6]:
# Schedules
schedules = standardize_team_column(schedules, "home_team")
schedules = standardize_team_column(schedules, "away_team")

# Rosters
rosters = standardize_team_column(rosters, "team")

# Player statistics
player_stats = standardize_team_column(player_stats, "recent_team")

# Participation
participation = standardize_team_column(participation, "possession_team")

# Snap counts
snap_counts = standardize_team_column(snap_counts, "team")
snap_counts = standardize_team_column(snap_counts, "opponent")

# Depth charts
depth_charts = standardize_team_column(depth_charts, "team")
depth_charts = standardize_team_column(depth_charts, "club_code")

# Draft picks
draft_picks = standardize_team_column(draft_picks, "team")

# Injuries
injuries = standardize_team_column(injuries, "team")

# Trades
trades = standardize_team_column(trades, "gave")
trades = standardize_team_column(trades, "received")

# Weekly rosters
weekly_rosters = standardize_team_column(weekly_rosters, "team")

print("Team abbreviations standardized.")

Team abbreviations standardized.


## Validate Team Identifiers

After standardization, team identifiers are reviewed across datasets to identify any unexpected abbreviations, historical aliases, or source specific naming conventions that still require attention.

In [7]:
team_checks = {
    "schedules_home": schedules.select("home_team").unique().sort("home_team"),
    "schedules_away": schedules.select("away_team").unique().sort("away_team"),
    "rosters": rosters.select("team").unique().sort("team"),
    "player_stats": player_stats.select("recent_team").unique().sort("recent_team"),
    "participation": participation.select("possession_team").unique().sort("possession_team"),
    "snap_counts_team": snap_counts.select("team").unique().sort("team"),
    "snap_counts_opponent": snap_counts.select("opponent").unique().sort("opponent"),
    "draft_picks": draft_picks.select("team").unique().sort("team"),
    "injuries": injuries.select("team").unique().sort("team"),
    "weekly_rosters": weekly_rosters.select("team").unique().sort("team"),
}

for name, values in team_checks.items():
    print(f"\n{name}")
    print(values.to_series().to_list())


schedules_home
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

schedules_away
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

rosters
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

player_stats
['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']

participation
[None, '', 'ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'C

# Standardize Season and Game Type

The projection model is primarily trained on regular season NFL performance.

Before filtering any dataset, game type fields are inspected to understand how regular season, postseason, and preseason records are represented across sources.

In [8]:
game_type_checks = {}

for name, df in {
    "schedules": schedules,
    "player_stats": player_stats,
    "participation": participation,
    "snap_counts": snap_counts,
    "injuries": injuries,
    "weekly_rosters": weekly_rosters,
}.items():

    relevant_columns = [
        col for col in ["game_type", "season_type"]
        if col in df.columns
    ]

    for col in relevant_columns:
        values = (
            df.select(col)
            .unique()
            .sort(col)
            .to_series()
            .to_list()
        )

        game_type_checks[f"{name}.{col}"] = values

for name, values in game_type_checks.items():
    print(f"{name}: {values}")

schedules.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
player_stats.season_type: ['REG']
snap_counts.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
injuries.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']
injuries.season_type: [None, 'POST', 'REG']
weekly_rosters.game_type: ['CON', 'DIV', 'REG', 'SB', 'WC']


## Create Regular Season Datasets

The primary projection features will be based on regular season performance so that teams are evaluated over comparable portions of each season.

Postseason records remain preserved in the raw datasets and can still be used later for separate analysis if needed.

In [9]:
schedules_reg = schedules.filter(
    pl.col("game_type") == "REG"
)

player_stats_reg = player_stats.filter(
    pl.col("season_type") == "REG"
)

snap_counts_reg = snap_counts.filter(
    pl.col("game_type") == "REG"
)

injuries_reg = injuries.filter(
    pl.col("game_type") == "REG"
)

weekly_rosters_reg = weekly_rosters.filter(
    pl.col("game_type") == "REG"
)

print("Regular-season datasets created.")
print()
print(f"Schedules:       {schedules.height:,} -> {schedules_reg.height:,}")
print(f"Player Stats:    {player_stats.height:,} -> {player_stats_reg.height:,}")
print(f"Snap Counts:     {snap_counts.height:,} -> {snap_counts_reg.height:,}")
print(f"Injuries:        {injuries.height:,} -> {injuries_reg.height:,}")
print(f"Weekly Rosters:  {weekly_rosters.height:,} -> {weekly_rosters_reg.height:,}")

Regular-season datasets created.

Schedules:       3,028 -> 2,895
Player Stats:    21,377 -> 21,377
Snap Counts:     276,948 -> 264,774
Injuries:        60,788 -> 58,449
Weekly Rosters:  498,381 -> 475,749


## Validate Regular Season Schedule Coverage

Regular season game counts are checked by season to confirm that the schedule dataset has complete historical coverage.

The NFL expanded from a 16 game schedule to a 17 game schedule beginning in 2021, so expected league wide game totals differ across the modeling period.

In [10]:
schedule_counts = (
    schedules_reg
    .group_by("season")
    .agg(
        pl.len().alias("games"),
        pl.col("home_team").n_unique().alias("home_teams"),
        pl.col("away_team").n_unique().alias("away_teams"),
        pl.col("week").min().alias("min_week"),
        pl.col("week").max().alias("max_week")
    )
    .sort("season")
)

schedule_counts

season,games,home_teams,away_teams,min_week,max_week
i32,u32,u32,u32,i32,i32
2015,256,32,32,1,17
2016,256,32,32,1,17
2017,256,32,32,1,17
2018,256,32,32,1,17
2019,256,32,32,1,17
…,…,…,…,…,…
2021,272,32,32,1,18
2022,271,32,32,1,18
2023,272,32,32,1,18


In [11]:
EXPECTED_REGULAR_SEASON_GAMES = {
    2015: 256,
    2016: 256,
    2017: 256,
    2018: 256,
    2019: 256,
    2020: 256,
    2021: 272,
    2022: 271,  # BUF-CIN Week 17 game was canceled
    2023: 272,
    2024: 272,
    2025: 272,
}

schedule_validation = (
    schedule_counts
    .with_columns(
        pl.col("season")
        .replace_strict(
            EXPECTED_REGULAR_SEASON_GAMES,
            default=None
        )
        .alias("expected_games")
    )
    .with_columns(
        (pl.col("games") == pl.col("expected_games"))
        .alias("game_count_valid"),

        (
            (pl.col("home_teams") == 32) &
            (pl.col("away_teams") == 32)
        ).alias("team_count_valid")
    )
)

schedule_validation

season,games,home_teams,away_teams,min_week,max_week,expected_games,game_count_valid,team_count_valid
i32,u32,u32,u32,i32,i32,i64,bool,bool
2015,256,32,32,1,17,256,true,true
2016,256,32,32,1,17,256,true,true
2017,256,32,32,1,17,256,true,true
2018,256,32,32,1,17,256,true,true
2019,256,32,32,1,17,256,true,true
…,…,…,…,…,…,…,…,…
2021,272,32,32,1,18,272,true,true
2022,271,32,32,1,18,271,true,true
2023,272,32,32,1,18,272,true,true


In [12]:
assert schedule_validation["game_count_valid"].all()
assert schedule_validation["team_count_valid"].all()

print("Regular season schedule validation passed.")

Regular season schedule validation passed.


# Clean Schedule Data

The regular season schedule is reduced to a standardized game level table.

This table will serve as the canonical game reference for downstream joins and will preserve game identity, teams, scores, location, rest, and schedule context.

In [13]:
schedule_clean = (
    schedules_reg
    .select([
        "game_id",
        "season",
        "week",
        "gameday",
        "weekday",
        "gametime",
        "away_team",
        "away_score",
        "home_team",
        "home_score",
        "location",
        "result",
        "total",
        "overtime",
        "away_rest",
        "home_rest",
    ])
    .sort(["season", "week", "gameday", "game_id"])
)

print(schedule_clean.shape)
schedule_clean.head(10)

(2895, 16)


game_id,season,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,away_rest,home_rest
str,i32,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,i32,i32
"""2015_01_PIT_NE""",2015,1,"""2015-09-10""","""Thursday""","""20:30""","""PIT""",21,"""NE""",28,"""Home""",7,49,0,7,7
"""2015_01_BAL_DEN""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""BAL""",13,"""DEN""",19,"""Home""",6,32,0,7,7
"""2015_01_CAR_JAX""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CAR""",20,"""JAX""",9,"""Home""",-11,29,0,7,7
"""2015_01_CIN_OAK""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""CIN""",33,"""LV""",13,"""Home""",-20,46,0,7,7
"""2015_01_CLE_NYJ""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CLE""",10,"""NYJ""",31,"""Home""",21,41,0,7,7
"""2015_01_DET_SD""",2015,1,"""2015-09-13""","""Sunday""","""16:05""","""DET""",28,"""LAC""",33,"""Home""",5,61,0,7,7
"""2015_01_GB_CHI""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""GB""",31,"""CHI""",23,"""Home""",-8,54,0,7,7
"""2015_01_IND_BUF""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""IND""",14,"""BUF""",27,"""Home""",13,41,0,7,7
"""2015_01_KC_HOU""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""KC""",27,"""HOU""",20,"""Home""",-7,47,0,7,7


In [14]:
duplicate_game_ids = (
    schedule_clean
    .group_by("game_id")
    .len()
    .filter(pl.col("len") > 1)
)

print(f"Duplicate game IDs: {duplicate_game_ids.height}")
print(f"Null game IDs: {schedule_clean['game_id'].null_count()}")

Duplicate game IDs: 0
Null game IDs: 0


## Schedule Missing Value Audit

The cleaned schedule is checked for missing values before being saved.

Missing values are reviewed rather than automatically filled so that source specific issues are not hidden during cleaning.

In [15]:
schedule_nulls = pl.DataFrame({
    "column": schedule_clean.columns,
    "null_count": [
        schedule_clean[col].null_count()
        for col in schedule_clean.columns
    ]
}).filter(
    pl.col("null_count") > 0
)

schedule_nulls

column,null_count
str,i64


In [16]:
schedule_clean_output = PROCESSED_DIR / "schedule_clean.parquet"

schedule_clean.write_parquet(schedule_clean_output)

print(f"Saved {schedule_clean.height:,} cleaned games to:")
print(schedule_clean_output)

Saved 2,895 cleaned games to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\schedule_clean.parquet


# Clean Season Roster Data

Season roster data provides the foundation for identifying players, positions, experience, and team membership across seasons.

The cleaning process preserves stable player identifiers wherever possible and standardizes the fields that will later connect roster information with performance, snap count, injury, and personnel datasets.

In [17]:
roster_id_check = pl.DataFrame({
    "identifier": [
        "gsis_id",
        "espn_id",
        "pfr_id",
        "sportradar_id"
    ],
    "non_null": [
        rosters["gsis_id"].is_not_null().sum(),
        rosters["espn_id"].is_not_null().sum(),
        rosters["pfr_id"].is_not_null().sum(),
        rosters["sportradar_id"].is_not_null().sum(),
    ],
    "null": [
        rosters["gsis_id"].null_count(),
        rosters["espn_id"].null_count(),
        rosters["pfr_id"].null_count(),
        rosters["sportradar_id"].null_count(),
    ]
})

roster_id_check

identifier,non_null,null
str,i64,i64
"""gsis_id""",33184,11
"""espn_id""",21142,12053
"""pfr_id""",16126,17069
"""sportradar_id""",21838,11357


In [18]:
missing_gsis = (
    rosters
    .filter(pl.col("gsis_id").is_null())
    .select([
        "season",
        "week",
        "team",
        "full_name",
        "position",
        "depth_chart_position",
        "status",
        "espn_id",
        "pfr_id",
        "sportradar_id"
    ])
    .sort(["season", "team", "full_name"])
)

print(f"Roster records missing GSIS ID: {missing_gsis.height}")

missing_gsis

Roster records missing GSIS ID: 11


season,week,team,full_name,position,depth_chart_position,status,espn_id,pfr_id,sportradar_id
i32,i32,str,str,str,str,str,str,str,str
2015,18,"""NE""","""Dick Conn""","""DB""",null,null,null,null,null
2018,21,"""NE""","""Darren Andrews""","""WR""","""WR""","""RSN""",null,null,null
2019,17,"""ATL""","""Tavonn Salter""","""WR""","""WR""","""DEV""",null,null,null
2020,19,"""GB""","""J.J. Molson""","""K""","""K""","""DEV""",null,null,null
2021,21,"""KC""","""Mark Vital""","""TE""","""TE""","""DEV""",null,null,null
…,…,…,…,…,…,…,…,…,…
2023,18,"""DEN""","""Durell Nchami""","""LB""","""LB""","""DEV""",null,null,null
2024,18,"""NO""","""Tra Fluellen""","""DB""","""DB""","""DEV""",null,null,null
2025,18,"""HOU""","""Layne Pryor""","""TE""","""TE""","""DEV""",null,null,null


## Player Identity Rules

`gsis_id` is used as the primary player identifier throughout the project because it provides nearly complete coverage and is shared across multiple NFL datasets.

Players without a GSIS ID are retained rather than removed or assigned an inferred identifier. These records may remain unmatched in downstream player level joins.

Player names are preserved for display and validation purposes but are not treated as unique identifiers.

In [19]:
rosters_clean = (
    rosters
    .select([
        "season",
        "team",
        "gsis_id",
        "full_name",
        "football_name",
        "position",
        "depth_chart_position",
        "status",
        "years_exp",
        "birth_date",
        "height",
        "weight",
        "college",
        "espn_id",
        "pfr_id",
        "sportradar_id",
    ])
    .sort([
        "season",
        "team",
        "gsis_id"
    ])
)

print(rosters_clean.shape)
rosters_clean.head(10)

(33195, 16)


season,team,gsis_id,full_name,football_name,position,depth_chart_position,status,years_exp,birth_date,height,weight,college,espn_id,pfr_id,sportradar_id
i32,str,str,str,str,str,str,str,i32,date,f64,i32,str,str,str,str
2015,"""ARI""","""00-0019435""","""Mike Leach""","""Mike""","""LS""",null,"""ACT""",15,1976-10-18,74.0,235,"""William & Mary""",null,null,null
2015,"""ARI""","""00-0021146""","""Dwight Freeney""","""Dwight""","""OLB""",null,"""ACT""",13,1980-02-19,73.0,268,"""Syracuse""",null,null,null
2015,"""ARI""","""00-0021429""","""Carson Palmer""","""Carson""","""QB""",null,"""ACT""",12,1979-12-27,77.0,235,"""USC""","""4459""","""PalmCa00""","""57ad34b3-f60d-4b2d-9e01-3cb5a9…"
2015,"""ARI""","""00-0021998""","""Cory Redding""","""Cory""","""DT""",null,"""RES""",12,1980-11-15,76.0,318,"""Texas""",null,null,null
2015,"""ARI""","""00-0022695""","""Jason Babin""","""Jason""","""OLB""",null,"""ACT""",11,1980-05-24,75.0,267,"""Western Michigan""",null,null,null
2015,"""ARI""","""00-0022921""","""Larry Fitzgerald""","""Larry""","""WR""",null,"""ACT""",11,1983-08-31,75.0,218,"""Pittsburgh""","""5528""","""FitzLa00""","""b6a61b38-5cfa-46eb-b1c5-b0255d…"
2015,"""ARI""","""00-0024306""","""Frostee Rucker""","""Frostee""","""DT""",null,"""ACT""",9,1983-09-14,75.0,280,"""USC""","""9677""","""RuckFr99""","""36648f14-5fe5-40f3-ade1-ef53c8…"
2015,"""ARI""","""00-0025430""","""Drew Stanton""","""Drew""","""QB""",null,"""ACT""",8,1984-05-07,75.0,243,"""Michigan State""","""10487""","""StanDr00""","""22fb2b54-4936-4e8a-a48d-62096c…"
2015,"""ARI""","""00-0025433""","""LaMarr Woodley""","""LaMarr""","""LB""",null,"""RES""",8,1984-11-03,74.0,265,"""Michigan""",null,null,null


In [20]:
roster_duplicates = (
    rosters_clean
    .filter(pl.col("gsis_id").is_not_null())
    .group_by([
        "season",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate player-season-team combinations: {roster_duplicates.height}")

roster_duplicates.head(20)

Duplicate player-season-team combinations: 0


season,team,gsis_id,len
i32,str,str,u32


## Validate Season Roster Coverage

Season roster coverage is checked by season to confirm that all 32 NFL teams are represented before the cleaned dataset is saved.

In [21]:
roster_coverage = (
    rosters_clean
    .group_by("season")
    .agg([
        pl.len().alias("roster_records"),
        pl.col("team").n_unique().alias("teams"),
        pl.col("gsis_id").null_count().alias("missing_gsis_ids"),
    ])
    .sort("season")
)

roster_coverage

season,roster_records,teams,missing_gsis_ids
i32,u32,u32,u32
2015,2190,32,1
2016,3061,32,0
2017,3082,32,0
2018,3142,32,1
2019,3114,32,1
…,…,…,…
2021,2961,32,1
2022,3134,32,1
2023,3090,32,1


In [22]:
rosters_clean_output = PROCESSED_DIR / "rosters_clean.parquet"

rosters_clean.write_parquet(rosters_clean_output)

print(f"Saved {rosters_clean.height:,} cleaned roster records to:")
print(rosters_clean_output)

Saved 33,195 cleaned roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\rosters_clean.parquet


# Clean Player Statistics

Player statistics provide the individual performance data used later for quarterback projections, skill position evaluation, and position group analysis.

The cleaning process focuses on preserving player identity, validating season coverage, identifying duplicate player season records, and retaining the statistical fields needed for downstream modeling.

In [23]:
player_stats_id_check = pl.DataFrame({
    "field": [
        "player_id",
        "player_name",
        "player_display_name",
        "recent_team"
    ],
    "null_count": [
        player_stats_reg["player_id"].null_count(),
        player_stats_reg["player_name"].null_count(),
        player_stats_reg["player_display_name"].null_count(),
        player_stats_reg["recent_team"].null_count(),
    ]
})

player_stats_id_check

field,null_count
str,i64
"""player_id""",11
"""player_name""",6
"""player_display_name""",11
"""recent_team""",0


In [24]:
player_stats_duplicates = (
    player_stats_reg
    .filter(pl.col("player_id").is_not_null())
    .group_by([
        "season",
        "player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate player-season combinations: "
    f"{player_stats_duplicates.height}"
)

player_stats_duplicates.head(20)

Duplicate player-season combinations: 0


season,player_id,len
i32,str,u32


## Validate Player Statistic Identifiers

Player ID is used as the primary identifier for the season level player statistics dataset.

Records without a player ID are inspected before any decision is made to remove or retain them. Player names are preserved for display purposes but are not treated as unique identifiers.

In [25]:
missing_player_ids = (
    player_stats_reg
    .filter(pl.col("player_id").is_null())
    .select([
        "season",
        "recent_team",
        "player_name",
        "player_display_name",
        "position",
        "position_group",
        "games",
        "attempts",
        "carries",
        "targets",
    ])
    .sort([
        "season",
        "recent_team",
        "player_display_name"
    ])
)

print(f"Player stat records missing player ID: {missing_player_ids.height}")

missing_player_ids

Player stat records missing player ID: 11


season,recent_team,player_name,player_display_name,position,position_group,games,attempts,carries,targets
i32,str,str,str,str,str,i32,i32,i32,i32
2015,"""GB""","""Team""",null,null,null,256,0,0,0
2016,"""DET""","""Team""",null,null,null,256,0,0,0
2017,"""JAX""","""Team""",null,null,null,256,0,0,0
2018,"""IND""","""R.Rodgers""",null,null,null,256,0,0,0
2019,"""SEA""","""Team""",null,null,null,256,0,0,0
…,…,…,…,…,…,…,…,…,…
2021,"""LV""",null,null,null,null,272,0,0,0
2022,"""DET""",null,null,null,null,271,0,0,0
2023,"""MIA""",null,null,null,null,272,0,0,0


## Remove Non-Player Aggregate Records

A small number of records without player IDs represent league or team level aggregate rows rather than individual players.

These records contain season-wide game counts and no meaningful player level usage. Because the player statistics dataset is intended to contain one record per individual player and season, these aggregate records are excluded from the cleaned dataset.

In [26]:
player_stats_clean = (
    player_stats_reg
    .filter(pl.col("player_id").is_not_null())
)

print(f"Original records: {player_stats_reg.height:,}")
print(f"Removed non-player records: {player_stats_reg.height - player_stats_clean.height:,}")
print(f"Clean player records: {player_stats_clean.height:,}")

Original records: 21,377
Removed non-player records: 11
Clean player records: 21,366


In [27]:
print(f"Missing player IDs: {player_stats_clean['player_id'].null_count()}")
print(f"Missing display names: {player_stats_clean['player_display_name'].null_count()}")

player_stats_clean.group_by("season").agg(
    pl.len().alias("players")
).sort("season")

Missing player IDs: 0
Missing display names: 0


season,players
i32,u32
2015,1845
2016,1855
2017,1868
2018,1883
2019,1888
…,…
2021,2081
2022,2006
2023,1942


In [28]:
player_stats_clean_output = PROCESSED_DIR / "player_stats_clean.parquet"

player_stats_clean.write_parquet(player_stats_clean_output)

print(f"Saved {player_stats_clean.height:,} cleaned player stat records to:")
print(player_stats_clean_output)

Saved 21,366 cleaned player stat records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\player_stats_clean.parquet


# Clean Snap Count Data

Snap counts provide player level playing time at the game level.

This dataset will later support returning production, starter continuity, offensive line stability, defensive front rotations, and the importance of roster additions and departures.

The cleaning process focuses on player identity, game identity, duplicate records, and missing snap information.

In [29]:
snap_id_check = pl.DataFrame({
    "field": [
        "game_id",
        "pfr_player_id",
        "player",
        "team",
        "opponent"
    ],
    "null_count": [
        snap_counts_reg["game_id"].null_count(),
        snap_counts_reg["pfr_player_id"].null_count(),
        snap_counts_reg["player"].null_count(),
        snap_counts_reg["team"].null_count(),
        snap_counts_reg["opponent"].null_count(),
    ]
})

snap_id_check

field,null_count
str,i64
"""game_id""",0
"""pfr_player_id""",0
"""player""",0
"""team""",0
"""opponent""",0


In [30]:
snap_duplicates = (
    snap_counts_reg
    .group_by([
        "game_id",
        "team",
        "pfr_player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate player-game combinations: {snap_duplicates.height}")

snap_duplicates.head(20)

Duplicate player-game combinations: 0


game_id,team,pfr_player_id,len
str,str,str,u32


In [31]:
snap_counts_clean = (
    snap_counts_reg
    .select([
        "game_id",
        "season",
        "week",
        "player",
        "pfr_player_id",
        "position",
        "team",
        "opponent",
        "offense_snaps",
        "offense_pct",
        "defense_snaps",
        "defense_pct",
        "st_snaps",
        "st_pct",
    ])
    .sort([
        "season",
        "week",
        "game_id",
        "team",
        "pfr_player_id"
    ])
)

print(snap_counts_clean.shape)
snap_counts_clean.head(10)

(264774, 14)


game_id,season,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
str,i32,i32,str,str,str,str,str,f64,f64,f64,f64,f64,f64
"""2015_01_BAL_DEN""",2015,1,"""Kamar Aiken""","""AikeKa00""","""WR""","""BAL""","""DEN""",44.0,0.76,0.0,0.0,1.0,0.03
"""2015_01_BAL_DEN""",2015,1,"""Javorius Allen""","""AlleJa01""","""RB""","""BAL""","""DEN""",13.0,0.22,0.0,0.0,6.0,0.21
"""2015_01_BAL_DEN""",2015,1,"""Kyle Arrington""","""ArriKy00""","""CB""","""BAL""","""DEN""",0.0,0.0,39.0,0.56,0.0,0.0
"""2015_01_BAL_DEN""",2015,1,"""Nick Boyle""","""BoylNi00""","""TE""","""BAL""","""DEN""",8.0,0.14,0.0,0.0,11.0,0.38
"""2015_01_BAL_DEN""",2015,1,"""Terrence Brooks""","""BrooTe00""","""FS""","""BAL""","""DEN""",0.0,0.0,0.0,0.0,19.0,0.66
"""2015_01_BAL_DEN""",2015,1,"""Arthur Brown""","""BrowAr00""","""LB""","""BAL""","""DEN""",0.0,0.0,0.0,0.0,14.0,0.48
"""2015_01_BAL_DEN""",2015,1,"""Marlon Brown""","""BrowMa00""","""WR""","""BAL""","""DEN""",34.0,0.59,0.0,0.0,3.0,0.1
"""2015_01_BAL_DEN""",2015,1,"""Michael Campanaro""","""CampMi02""","""WR""","""BAL""","""DEN""",11.0,0.19,0.0,0.0,6.0,0.21
"""2015_01_BAL_DEN""",2015,1,"""Chris Canty""","""CantCh21""","""DE""","""BAL""","""DEN""",0.0,0.0,42.0,0.6,5.0,0.17


In [32]:
snap_value_check = pl.DataFrame({
    "field": [
        "offense_snaps",
        "offense_pct",
        "defense_snaps",
        "defense_pct",
        "st_snaps",
        "st_pct"
    ],
    "null_count": [
        snap_counts_clean[col].null_count()
        for col in [
            "offense_snaps",
            "offense_pct",
            "defense_snaps",
            "defense_pct",
            "st_snaps",
            "st_pct"
        ]
    ]
})

snap_value_check

field,null_count
str,i64
"""offense_snaps""",0
"""offense_pct""",0
"""defense_snaps""",0
"""defense_pct""",0
"""st_snaps""",0
"""st_pct""",0


In [33]:
snap_pct_validation = pl.DataFrame({
    "field": [
        "offense_pct",
        "defense_pct",
        "st_pct"
    ],
    "min_value": [
        snap_counts_clean["offense_pct"].min(),
        snap_counts_clean["defense_pct"].min(),
        snap_counts_clean["st_pct"].min(),
    ],
    "max_value": [
        snap_counts_clean["offense_pct"].max(),
        snap_counts_clean["defense_pct"].max(),
        snap_counts_clean["st_pct"].max(),
    ]
})

snap_pct_validation

field,min_value,max_value
str,f64,f64
"""offense_pct""",0.0,1.0
"""defense_pct""",0.0,1.0
"""st_pct""",0.0,1.01


In [34]:
assert (
    snap_counts_clean["offense_pct"].min() >= 0
    and snap_counts_clean["offense_pct"].max() <= 1
)

assert (
    snap_counts_clean["defense_pct"].min() >= 0
    and snap_counts_clean["defense_pct"].max() <= 1
)

assert (
    snap_counts_clean["st_pct"].min() >= 0
    and snap_counts_clean["st_pct"].max() <= 1.01
)

print("Snap percentage validation passed.")

Snap percentage validation passed.


In [35]:
snap_counts_clean_output = PROCESSED_DIR / "snap_counts_clean.parquet"

snap_counts_clean.write_parquet(snap_counts_clean_output)

print(f"Saved {snap_counts_clean.height:,} cleaned snap count records to:")
print(snap_counts_clean_output)

Saved 264,774 cleaned snap count records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\snap_counts_clean.parquet


# Clean Participation and Personnel Data

Participation data provides play level information about offensive and defensive personnel, formations, box counts, pass rushers, pressure, and time to throw.

These fields will later support team level features describing offensive structure, pass protection, defensive pressure, front usage, and personnel tendencies.

Because participation data begins in 2016, features derived from this source will have a shorter historical window than the core play-by-play and schedule datasets.

In [36]:
participation_id_check = pl.DataFrame({
    "field": [
        "nflverse_game_id",
        "play_id",
        "possession_team"
    ],
    "null_count": [
        participation["nflverse_game_id"].null_count(),
        participation["play_id"].null_count(),
        participation["possession_team"].null_count(),
    ]
})

participation_id_check

field,null_count
str,i64
"""nflverse_game_id""",0
"""play_id""",0
"""possession_team""",4


In [37]:
participation_duplicates = (
    participation
    .group_by([
        "nflverse_game_id",
        "play_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate game-play combinations: "
    f"{participation_duplicates.height}"
)

participation_duplicates.head(20)

Duplicate game-play combinations: 0


nflverse_game_id,play_id,len
str,f64,u32


## Restrict Participation Data to Regular Season Games

The participation dataset does not contain a dedicated season type field.

Regular season participation records are therefore identified by matching `nflverse_game_id` against the validated regular season game IDs contained in the cleaned schedule dataset. This ensures postseason and other non-regular season records are excluded consistently.

In [38]:
regular_season_game_ids = (
    schedule_clean
    .select(
        pl.col("game_id").alias("nflverse_game_id")
    )
    .unique()
)

participation_reg = (
    participation
    .join(
        regular_season_game_ids,
        on="nflverse_game_id",
        how="inner"
    )
)

print(f"Original participation records: {participation.height:,}")
print(f"Regular season participation records: {participation_reg.height:,}")
print(f"Removed records: {participation.height - participation_reg.height:,}")

Original participation records: 478,989
Regular season participation records: 457,830
Removed records: 21,159


In [39]:
participation_reg = participation_reg.with_columns(
    pl.col("nflverse_game_id")
    .str.slice(0, 4)
    .cast(pl.Int32)
    .alias("season")
)

In [40]:
participation_season_check = (
    participation_reg
    .group_by("season")
    .agg([
        pl.len().alias("plays"),
        pl.col("nflverse_game_id").n_unique().alias("games"),
        pl.col("possession_team").n_unique().alias("teams"),
    ])
    .sort("season")
)

participation_season_check

season,plays,games,teams
i32,u32,u32,u32
2016,46468,256,33
2017,46000,256,33
2018,45850,256,33
2019,46091,256,33
2020,46189,256,33
2021,48416,272,34
2022,47844,271,34
2023,44057,272,32
2024,43866,272,32


In [41]:
participation_team_codes = (
    participation_reg
    .filter(
        pl.col("possession_team").is_not_null() &
        (pl.col("possession_team") != "")
    )
    .group_by("season")
    .agg(
        pl.col("possession_team")
        .unique()
        .sort()
        .alias("team_codes")
    )
    .sort("season")
)

participation_team_codes

season,team_codes
i32,list[str]
2016,"[""ARI"", ""ATL"", … ""WAS""]"
2017,"[""ARI"", ""ATL"", … ""WAS""]"
2018,"[""ARI"", ""ATL"", … ""WAS""]"
2019,"[""ARI"", ""ATL"", … ""WAS""]"
2020,"[""ARI"", ""ATL"", … ""WAS""]"
2021,"[""ARI"", ""ATL"", … ""WAS""]"
2022,"[""ARI"", ""ATL"", … ""WAS""]"
2023,"[""ARI"", ""ATL"", … ""WAS""]"
2024,"[""ARI"", ""ATL"", … ""WAS""]"


In [42]:
VALID_TEAMS = [
    "ARI", "ATL", "BAL", "BUF", "CAR", "CHI", "CIN", "CLE",
    "DAL", "DEN", "DET", "GB", "HOU", "IND", "JAX", "KC",
    "LA", "LAC", "LV", "MIA", "MIN", "NE", "NO", "NYG",
    "NYJ", "PHI", "PIT", "SEA", "SF", "TB", "TEN", "WAS"
]

participation_bad_team_codes = (
    participation_reg
    .filter(
        pl.col("possession_team").is_not_null() &
        (pl.col("possession_team") != "") &
        (~pl.col("possession_team").is_in(VALID_TEAMS))
    )
    .group_by([
        "season",
        "possession_team"
    ])
    .len()
    .sort([
        "season",
        "possession_team"
    ])
)

participation_bad_team_codes

season,possession_team,len
i32,str,u32


In [43]:
participation_missing_team = (
    participation_reg
    .filter(
        pl.col("possession_team").is_null() |
        (pl.col("possession_team") == "")
    )
    .select([
        "season",
        "nflverse_game_id",
        "play_id",
        "possession_team",
        "offense_formation",
        "offense_personnel",
        "defense_personnel",
        "number_of_pass_rushers",
        "time_to_throw",
        "was_pressure"
    ])
)

print(
    f"Missing/blank possession team records: "
    f"{participation_missing_team.height}"
)

participation_missing_team

Missing/blank possession team records: 27863


season,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,defense_personnel,number_of_pass_rushers,time_to_throw,was_pressure
i32,str,f64,str,str,str,str,i32,f64,bool
2016,"""2016_01_CAR_DEN""",1.0,"""""",null,null,null,null,null,null
2016,"""2016_01_CAR_DEN""",327.0,"""""",null,null,null,null,null,null
2016,"""2016_01_CAR_DEN""",410.0,"""""",null,null,null,null,null,null
2016,"""2016_01_CAR_DEN""",824.0,"""""",null,null,null,null,null,null
2016,"""2016_01_CAR_DEN""",1061.0,"""""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…
2022,"""2022_18_ARI_SF""",2928.0,"""""",null,null,null,null,null,null
2022,"""2022_18_ARI_SF""",3059.0,"""""",null,null,null,null,null,null
2022,"""2022_18_ARI_SF""",3520.0,"""""",null,null,null,null,null,null


## Remove Unusable Participation Records

Participation records without an identifiable possession team are excluded from the cleaned dataset.

These records contain almost no usable formation, pressure, pass rush, or box count information and cannot be reliably assigned to an offensive team. A small number contain personnel strings, but without a possession team they cannot support team level feature engineering.

In [44]:
participation_clean = (
    participation_reg
    .filter(
        pl.col("possession_team").is_not_null() &
        (pl.col("possession_team") != "")
    )
)

print(f"Regular season records: {participation_reg.height:,}")
print(f"Removed unusable records: {participation_reg.height - participation_clean.height:,}")
print(f"Clean participation records: {participation_clean.height:,}")

Regular season records: 457,830
Removed unusable records: 27,863
Clean participation records: 429,967


In [45]:
participation_field_coverage = pl.DataFrame({
    "field": [
        "offense_formation",
        "offense_personnel",
        "defenders_in_box",
        "defense_personnel",
        "number_of_pass_rushers",
        "time_to_throw",
        "was_pressure"
    ],
    "non_null": [
        participation_clean[col].is_not_null().sum()
        for col in [
            "offense_formation",
            "offense_personnel",
            "defenders_in_box",
            "defense_personnel",
            "number_of_pass_rushers",
            "time_to_throw",
            "was_pressure"
        ]
    ]
}).with_columns(
    (
        pl.col("non_null") / participation_clean.height
    ).alias("coverage_pct")
)

participation_field_coverage

field,non_null,coverage_pct
str,i64,f64
"""offense_formation""",342369,0.796268
"""offense_personnel""",379398,0.882389
"""defenders_in_box""",372990,0.867485
"""defense_personnel""",379398,0.882389
"""number_of_pass_rushers""",268845,0.625269
"""time_to_throw""",182365,0.424137
"""was_pressure""",256628,0.596855


## Preserve Participation Feature Coverage

Missing participation values are retained as null rather than replaced with zero because many fields apply only to specific play types or situations.

The cleaned participation dataset preserves personnel, formation, pressure, timing, and player participation information so feature selection and aggregation can be performed later during feature engineering.

In [46]:
participation_clean = (
    participation_clean
    .select([
        "season",
        "nflverse_game_id",
        "play_id",
        "possession_team",
        "offense_formation",
        "offense_personnel",
        "defenders_in_box",
        "defense_personnel",
        "number_of_pass_rushers",
        "players_on_play",
        "offense_players",
        "defense_players",
        "n_offense",
        "n_defense",
        "ngs_air_yards",
        "time_to_throw",
        "was_pressure",
        "route",
        "defense_man_zone_type",
        "defense_coverage_type",
        "offense_names",
        "defense_names",
        "offense_positions",
        "defense_positions",
        "offense_numbers",
        "defense_numbers",
    ])
    .sort([
        "season",
        "nflverse_game_id",
        "play_id"
    ])
)

print(participation_clean.shape)
participation_clean.head(10)

(429967, 26)


season,nflverse_game_id,play_id,possession_team,offense_formation,offense_personnel,defenders_in_box,defense_personnel,number_of_pass_rushers,players_on_play,offense_players,defense_players,n_offense,n_defense,ngs_air_yards,time_to_throw,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
i32,str,f64,str,str,str,i32,str,i32,str,str,str,i32,i32,f64,f64,bool,str,str,str,str,str,str,str,str,str
2016,"""2016_01_BUF_BAL""",36.0,"""BUF""",null,null,null,null,null,"""40078;40151;37977;36060;40494;…","""00-0030041;00-0030073;00-00297…","""00-0029892;00-0029895;00-00284…",10,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",58.0,"""BAL""","""I_FORM""","""2 RB, 1 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",null,"""38540;41302;40078;35553;38582;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",85.0,"""BAL""","""SHOTGUN""","""2 RB, 1 TE, 2 WR""",6,"""4 DL, 3 LB, 4 DB""",3,"""41302;40078;38540;35553;43295;…","""00-0029892;00-0027714;00-00329…","""00-0031171;00-0029542;00-00295…",11,11,-4.53,2.404,false,"""FLAT""",null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",109.0,"""BAL""","""SHOTGUN""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",null,"""40078;38540;41302;35553;43295;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",130.0,"""BAL""","""I_FORM""","""2 RB, 1 TE, 2 WR""",7,"""4 DL, 3 LB, 4 DB""",5,"""40078;38540;41302;35553;38582;…","""00-0029892;00-0027714;00-00329…","""00-0029542;00-0031171;00-00295…",11,11,1.71,2.093,false,"""FLAT""",null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",154.0,"""BAL""","""I_FORM""","""1 RB, 2 TE, 2 WR""",8,"""4 DL, 3 LB, 4 DB""",null,"""41302;38540;43295;38582;40053;…","""00-0032965;00-0029893;00-00262…","""00-0031171;00-0029542;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",175.0,"""BAL""",null,"""1 RB, 2 TE, 2 WR""",null,"""4 DL, 3 LB, 4 DB""",null,"""38540;41302;43295;38582;40053;…","""00-0032965;00-0029893;00-00262…","""00-0029542;00-0031171;00-00295…",11,11,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",198.0,"""BAL""",null,null,null,null,null,"""40078;35575;36060;37977;40494;…","""00-0029892;00-0027736;00-00275…","""00-0030041;00-0030073;00-00304…",10,10,null,null,null,null,null,null,null,null,null,null,null,null
2016,"""2016_01_BUF_BAL""",216.0,"""BUF""","""SHOTGUN""","""1 RB, 2 TE, 2 WR""",7,"""3 DL, 4 LB, 4 DB""",4,"""34479;40111;41277;37249;41230;…","""00-0027004;00-0030046;00-00281…","""00-0031170;00-0027560;00-00279…",11,10,1.61,2.537,false,"""ANGLE""",null,null,null,null,null,null,null,null


In [47]:
assert participation_clean.select(
    pl.struct(["nflverse_game_id", "play_id"]).n_unique()
).item() == participation_clean.height

assert (
    participation_clean
    .filter(
        pl.col("possession_team").is_null() |
        (pl.col("possession_team") == "")
    )
    .height == 0
)

assert (
    participation_clean
    .filter(~pl.col("possession_team").is_in(VALID_TEAMS))
    .height == 0
)

print("Participation validation passed.")

Participation validation passed.


In [48]:
participation_clean_output = PROCESSED_DIR / "participation_clean.parquet"

participation_clean.write_parquet(participation_clean_output)

print(f"Saved {participation_clean.height:,} cleaned participation records to:")
print(participation_clean_output)

Saved 429,967 cleaned participation records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\participation_clean.parquet


# Clean Injury Data

Injury reports provide weekly information about player availability, reported injuries, game status, and practice participation.

These records will later be combined with roster, snap count, and player information to create team level availability features. This allows the model to distinguish between injuries to high usage players and injuries to players with limited roles.

The raw injury data is first restricted to the regular season and standardized before any injury impact features are created.

In [49]:
injury_key_check = (
    injuries_reg
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(f"Duplicate season-week-team-player combinations: {injury_key_check.height}")

injury_key_check.head(20)

Duplicate season-week-team-player combinations: 2


season,week,team,gsis_id,len
f64,f64,str,str,u32
2024.0,15.0,"""HOU""","""00-0039359""",2
2024.0,15.0,"""NYJ""","""00-0034270""",2


In [50]:
injury_identifier_check = pl.DataFrame({
    "field": [
        "gsis_id",
        "full_name",
        "team",
        "week"
    ],
    "null_count": [
        injuries_reg["gsis_id"].null_count(),
        injuries_reg["full_name"].null_count(),
        injuries_reg["team"].null_count(),
        injuries_reg["week"].null_count()
    ]
})

injury_identifier_check

field,null_count
str,i64
"""gsis_id""",0
"""full_name""",0
"""team""",0
"""week""",0


In [54]:
report_status_values = (
    injuries_reg
    .select("report_status")
    .drop_nulls()
    .unique()
    .sort("report_status")
)

report_status_values

report_status
str
"""Doubtful"""
"""Note"""
"""Out"""
"""Probable"""
"""Questionable"""


In [55]:
practice_status_values = (
    injuries_reg
    .select("practice_status")
    .drop_nulls()
    .unique()
    .sort("practice_status")
)

practice_status_values

practice_status
str
""" """
"""Did Not Participate In Practic…"
"""Full Participation in Practice"""
"""Limited Participation in Pract…"
"""Note"""
"""Out (Definitely Will Not Play)"""


In [56]:
injuries_reg.filter(
    (
        (pl.col("season") == 2024) &
        (pl.col("week") == 15) &
        (pl.col("team") == "HOU") &
        (pl.col("gsis_id") == "00-0039359")
    )
    |
    (
        (pl.col("season") == 2024) &
        (pl.col("week") == 15) &
        (pl.col("team") == "NYJ") &
        (pl.col("gsis_id") == "00-0034270")
    )
).sort([
    "team",
    "gsis_id",
    "date_modified"
])

season,game_type,team,week,gsis_id,position,full_name,first_name,last_name,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status,date_modified,season_type
f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,"datetime[μs, UTC]",str
2024.0,"""REG""","""HOU""",15.0,"""00-0039359""","""TE""","""Cade Stover""","""Cade""","""Stover""","""Illness""",null,"""Questionable""",null,null,""" """,2024-12-15 03:34:33 UTC,null
2024.0,"""REG""","""HOU""",15.0,"""00-0039359""","""TE""","""Cade Stover""","""Cade""","""Stover""","""Illness""",null,"""Out""",null,null,""" """,2024-12-15 14:17:06 UTC,null
2024.0,"""REG""","""NYJ""",15.0,"""00-0034270""","""TE""","""Tyler Conklin""","""Tyler""","""Conklin""","""Not injury related - personal …",null,"""Questionable""",null,null,""" """,2024-12-14 20:55:19 UTC,null
2024.0,"""REG""","""NYJ""",15.0,"""00-0034270""","""TE""","""Tyler Conklin""","""Tyler""","""Conklin""","""Not injury related - personal …",null,"""Out""",null,null,""" """,2024-12-15 13:57:00 UTC,null


In [57]:
injuries_reg.filter(
    (
        (pl.col("season") == 2024) &
        (pl.col("week") == 15) &
        (pl.col("team").is_in(["HOU", "NYJ"])) &
        (pl.col("gsis_id").is_in(["00-0039359", "00-0034270"]))
    )
).select([
    "season",
    "week",
    "team",
    "full_name",
    "gsis_id",
    "report_status",
    "practice_status",
    "date_modified"
]).sort([
    "team",
    "gsis_id",
    "date_modified"
])

season,week,team,full_name,gsis_id,report_status,practice_status,date_modified
f64,f64,str,str,str,str,str,"datetime[μs, UTC]"
2024.0,15.0,"""HOU""","""Cade Stover""","""00-0039359""","""Questionable""",""" """,2024-12-15 03:34:33 UTC
2024.0,15.0,"""HOU""","""Cade Stover""","""00-0039359""","""Out""",""" """,2024-12-15 14:17:06 UTC
2024.0,15.0,"""NYJ""","""Tyler Conklin""","""00-0034270""","""Questionable""",""" """,2024-12-14 20:55:19 UTC
2024.0,15.0,"""NYJ""","""Tyler Conklin""","""00-0034270""","""Out""",""" """,2024-12-15 13:57:00 UTC


In [58]:
injuries_clean = (
    injuries_reg
    .sort("date_modified")
    .unique(
        subset=[
            "season",
            "week",
            "team",
            "gsis_id"
        ],
        keep="last"
    )
    .sort([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

print(f"Original regular season injury records: {injuries_reg.height:,}")
print(f"Clean injury records: {injuries_clean.height:,}")
print(f"Updated records removed: {injuries_reg.height - injuries_clean.height:,}")

Original regular season injury records: 58,449
Clean injury records: 58,447
Updated records removed: 2


## Standardize Injury Status Fields

Blank injury and practice status strings are converted to null values so missing information is represented consistently.

Original injury terminology is otherwise preserved. Status categories are not collapsed or reinterpreted during cleaning because game designations and practice participation provide different types of information.

In [60]:
injuries_clean = injuries_clean.with_columns([
    pl.when(
        pl.col("report_status").str.strip_chars() == ""
    )
    .then(None)
    .otherwise(pl.col("report_status"))
    .alias("report_status"),

    pl.when(
        pl.col("practice_status").str.strip_chars() == ""
    )
    .then(None)
    .otherwise(pl.col("practice_status"))
    .alias("practice_status")
])

In [61]:
injury_text_columns = [
    "report_primary_injury",
    "report_secondary_injury",
    "practice_primary_injury",
    "practice_secondary_injury"
]

injuries_clean = injuries_clean.with_columns([
    pl.when(pl.col(col).str.strip_chars() == "")
    .then(None)
    .otherwise(pl.col(col))
    .alias(col)
    for col in injury_text_columns
])

In [62]:
print(f"Clean injury records: {injuries_clean.height:,}")
print(f"Remaining duplicates: {injury_duplicate_validation.height}")
print(f"Missing GSIS IDs: {injuries_clean['gsis_id'].null_count()}")
print(f"Missing teams: {injuries_clean['team'].null_count()}")

Clean injury records: 58,447
Remaining duplicates: 0
Missing GSIS IDs: 0
Missing teams: 0


In [63]:
injuries_clean_output = PROCESSED_DIR / "injuries_clean.parquet"

injuries_clean.write_parquet(injuries_clean_output)

print(f"Saved {injuries_clean.height:,} cleaned injury records to:")
print(injuries_clean_output)

Saved 58,447 cleaned injury records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\injuries_clean.parquet


# Clean Weekly Roster Data

Weekly roster data provides a week by week view of team personnel and player status.

This dataset will later support roster continuity, player movement, active roster availability, practice squad activity, and returning production features.

The cleaning process focuses on regular season records, player identity, roster status, and duplicate player-week-team combinations.

In [64]:
weekly_roster_id_check = pl.DataFrame({
    "field": [
        "season",
        "week",
        "team",
        "gsis_id",
        "full_name",
        "status"
    ],
    "null_count": [
        weekly_rosters_reg["season"].null_count(),
        weekly_rosters_reg["week"].null_count(),
        weekly_rosters_reg["team"].null_count(),
        weekly_rosters_reg["gsis_id"].null_count(),
        weekly_rosters_reg["full_name"].null_count(),
        weekly_rosters_reg["status"].null_count(),
    ]
})

weekly_roster_id_check

field,null_count
str,i64
"""season""",0
"""week""",0
"""team""",0
"""gsis_id""",121
"""full_name""",16
"""status""",18


In [65]:
weekly_roster_duplicates = (
    weekly_rosters_reg
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(
    f"Duplicate season-week-team-player combinations: "
    f"{weekly_roster_duplicates.height}"
)

weekly_roster_duplicates.head(20)

Duplicate season-week-team-player combinations: 1702


season,week,team,gsis_id,len
i32,i32,str,str,u32
2015,16,"""SF""","""00-0028409""",4
2015,7,"""BAL""","""00-0031687""",4
2015,11,"""BAL""","""00-0031687""",4
2015,13,"""SEA""","""00-0028409""",4
2015,5,"""MIA""","""00-0031687""",4
…,…,…,…,…
2015,13,"""BAL""","""00-0031687""",4
2015,3,"""MIA""","""00-0031687""",4
2015,10,"""BAL""","""00-0031687""",4


In [66]:
weekly_roster_status_values = (
    weekly_rosters_reg
    .select("status")
    .drop_nulls()
    .unique()
    .sort("status")
)

weekly_roster_status_values

status
str
"""ACT"""
"""CUT"""
"""DEV"""
"""E01"""
"""E14"""
…
"""TRC"""
"""TRD"""
"""TRT"""


In [67]:
duplicate_examples = (
    weekly_roster_duplicates
    .head(5)
    .select([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

weekly_rosters_reg.join(
    duplicate_examples,
    on=[
        "season",
        "week",
        "team",
        "gsis_id"
    ],
    how="inner"
).sort([
    "season",
    "week",
    "team",
    "gsis_id"
])

season,team,position,depth_chart_position,jersey_number,status,full_name,first_name,last_name,birth_date,height,weight,college,gsis_id,espn_id,sportradar_id,yahoo_id,rotowire_id,pff_id,pfr_id,fantasy_data_id,sleeper_id,years_exp,headshot_url,ngs_position,week,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number
i32,str,str,str,str,str,str,str,str,date,f64,i32,str,str,str,str,str,str,str,str,str,str,i32,str,str,i32,str,str,str,str,str,str,i32,i32,str,str
2015,"""MIA""","""RB""",null,"""27""","""ACT""","""Raheem Mostert""","""Dominique""","""Mostert""",1992-04-09,70.0,195,"""Purdue""","""00-0031687""","""2576414""","""b040e601-ec40-4757-bf3d-71bf64…","""28654""","""10604""","""9783""","""MostRa00""","""17217""","""2749""",0,"""https://static.www.nfl.com/ima…",null,5,"""REG""","""A01""","""Raheem""","""MOS822492""","""42718""","""32004d4f-5382-2492-b0f0-aec08c…",2015,2015,null,null
2015,"""MIA""","""RB""",null,"""27""","""TRT""","""Raheem Mostert""","""Dominique""","""Mostert""",1992-04-09,70.0,195,"""Purdue""","""00-0031687""","""2576414""","""b040e601-ec40-4757-bf3d-71bf64…","""28654""","""10604""","""9783""","""MostRa00""","""17217""","""2749""",0,"""https://static.www.nfl.com/ima…",null,5,"""REG""","""A01""","""Raheem""","""MOS822492""","""42718""","""32004d4f-5382-2492-b0f0-aec08c…",2015,2015,null,null
2015,"""MIA""","""RB""",null,"""27""","""TRC""","""Raheem Mostert""","""Dominique""","""Mostert""",1992-04-09,70.0,195,"""Purdue""","""00-0031687""","""2576414""","""b040e601-ec40-4757-bf3d-71bf64…","""28654""","""10604""","""9783""","""MostRa00""","""17217""","""2749""",0,"""https://static.www.nfl.com/ima…",null,5,"""REG""","""A01""","""Raheem""","""MOS822492""","""42718""","""32004d4f-5382-2492-b0f0-aec08c…",2015,2015,null,null
2015,"""MIA""","""RB""",null,"""27""","""TRD""","""Raheem Mostert""","""Dominique""","""Mostert""",1992-04-09,70.0,195,"""Purdue""","""00-0031687""","""2576414""","""b040e601-ec40-4757-bf3d-71bf64…","""28654""","""10604""","""9783""","""MostRa00""","""17217""","""2749""",0,"""https://static.www.nfl.com/ima…",null,5,"""REG""","""A01""","""Raheem""","""MOS822492""","""42718""","""32004d4f-5382-2492-b0f0-aec08c…",2015,2015,null,null
2015,"""BAL""","""RB""",null,"""43""","""ACT""","""Raheem Mostert""","""Dominique""","""Mostert""",1992-04-09,70.0,195,"""Purdue""","""00-0031687""","""2576414""","""b040e601-ec40-4757-bf3d-71bf64…","""28654""","""10604""","""9783""","""MostRa00""","""17217""","""2749""",0,"""https://static.www.nfl.com/ima…",null,7,"""REG""","""A01""","""Raheem""","""MOS822492""","""42718""","""32004d4f-5382-2492-b0f0-aec08c…",2015,2015,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2015,"""SEA""","""RB""",null,"""32""","""ACT""","""DuJuan Harris""","""DuJuan""","""Harris""",1988-09-03,68.0,203,"""Troy""","""00-0028409""","""14255""","""7ea2fcfd-0099-4e62-8f6e-efa019…","""25117""","""7658""","""6595""",null,"""12985""","""863""",4,"""https://static.www.nfl.com/ima…",null,13,"""REG""","""A01""","""DuJuan""","""HAR370058""","""37889""","""32004841-5237-0058-7cc5-53d0d7…",2011,2011,null,null
2015,"""SF""","""RB""",null,"""32""","""ACT""","""DuJuan Harris""","""DuJuan""","""Harris""",1988-09-03,68.0,203,"""Troy""","""00-0028409""","""14255""","""7ea2fcfd-0099-4e62-8f6e-efa019…","""25117""","""7658""","""6595""",null,"""12985""","""863""",4,"""https://static.www.nfl.com/ima…",null,16,"""REG""","""A01""","""DuJuan""","""HAR370058""","""37889""","""32004841-5237-0058-7cc5-53d0d7…",2011,2011,null,null
2015,"""SF""","""RB""",null,"""32""","""TRC""","""DuJuan Harris""","""DuJuan""","""Harris""",1988-09-03,68.0,203,"""Troy""","""00-0028409""","""14255""","""7ea2fcfd-0099-4e62-8f6e-efa019…","""25117""","""7658""","""6595""",null,"""12985""","""863""",4,"""https://static.www.nfl.com/ima…",null,16,"""REG""","""A01""","""DuJuan""","""HAR370058""","""37889""","""32004841-5237-0058-7cc5-53d0d7…",2011,2011,n

In [68]:
duplicate_detail = (
    weekly_rosters_reg
    .join(
        duplicate_examples,
        on=[
            "season",
            "week",
            "team",
            "gsis_id"
        ],
        how="inner"
    )
    .select([
        "season",
        "week",
        "team",
        "gsis_id",
        "full_name",
        "position",
        "depth_chart_position",
        "status",
        "status_description_abbr",
        "years_exp",
        "entry_year",
        "rookie_year",
        "draft_club"
    ])
    .sort([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

duplicate_detail

season,week,team,gsis_id,full_name,position,depth_chart_position,status,status_description_abbr,years_exp,entry_year,rookie_year,draft_club
i32,i32,str,str,str,str,str,str,str,i32,i32,i32,str
2015,5,"""MIA""","""00-0031687""","""Raheem Mostert""","""RB""",null,"""ACT""","""A01""",0,2015,2015,null
2015,5,"""MIA""","""00-0031687""","""Raheem Mostert""","""RB""",null,"""TRT""","""A01""",0,2015,2015,null
2015,5,"""MIA""","""00-0031687""","""Raheem Mostert""","""RB""",null,"""TRC""","""A01""",0,2015,2015,null
2015,5,"""MIA""","""00-0031687""","""Raheem Mostert""","""RB""",null,"""TRD""","""A01""",0,2015,2015,null
2015,7,"""BAL""","""00-0031687""","""Raheem Mostert""","""RB""",null,"""ACT""","""A01""",0,2015,2015,null
…,…,…,…,…,…,…,…,…,…,…,…,…
2015,13,"""SEA""","""00-0028409""","""DuJuan Harris""","""RB""",null,"""ACT""","""A01""",4,2011,2011,null
2015,16,"""SF""","""00-0028409""","""DuJuan Harris""","""RB""",null,"""ACT""","""A01""",4,2011,2011,null
2015,16,"""SF""","""00-0028409""","""DuJuan Harris""","""RB""",null,"""TRC""","""A01""",4,2011,2011,null


In [69]:
print(f"Rows before exact deduplication: {weekly_rosters_reg.height:,}")
print(f"Unique full rows: {weekly_rosters_reg.unique().height:,}")
print(
    f"Exact duplicate rows: "
    f"{weekly_rosters_reg.height - weekly_rosters_reg.unique().height:,}"
)

Rows before exact deduplication: 475,749
Unique full rows: 475,749
Exact duplicate rows: 0


In [70]:
weekly_roster_duplicate_summary = (
    weekly_rosters_reg
    .group_by([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
    .len()
    .group_by("season")
    .agg([
        pl.len().alias("player_week_records"),
        (pl.col("len") > 1).sum().alias("duplicate_player_weeks"),
        pl.col("len").max().alias("max_records_per_player_week")
    ])
    .with_columns(
        (
            pl.col("duplicate_player_weeks") /
            pl.col("player_week_records")
        ).alias("duplicate_pct")
    )
    .sort("season")
)

weekly_roster_duplicate_summary

season,player_week_records,duplicate_player_weeks,max_records_per_player_week,duplicate_pct
i32,u32,u32,u32,f64
2015,28823,1695,4,0.058807
2016,32944,0,1,0.0
2017,49210,0,1,0.0
2018,50113,0,1,0.0
2019,49561,0,1,0.0
…,…,…,…,…
2021,44538,1,2,0.000022
2022,44053,6,2,0.000136
2023,43545,0,1,0.0


In [71]:
duplicate_status_summary = (
    weekly_rosters_reg
    .join(
        weekly_roster_duplicates.select([
            "season",
            "week",
            "team",
            "gsis_id"
        ]),
        on=[
            "season",
            "week",
            "team",
            "gsis_id"
        ],
        how="inner"
    )
    .group_by("status")
    .len()
    .sort("len", descending=True)
)

duplicate_status_summary

status,len
str,u32
"""ACT""",1401
"""TRC""",936
"""TRD""",691
"""TRT""",186
"""RES""",101
"""CUT""",100
"""DEV""",93


In [72]:
weekly_roster_missing_ids = (
    weekly_rosters_reg
    .filter(pl.col("gsis_id").is_null())
    .select([
        "season",
        "week",
        "team",
        "full_name",
        "football_name",
        "position",
        "depth_chart_position",
        "status",
        "espn_id",
        "pfr_id",
        "sportradar_id"
    ])
    .sort([
        "season",
        "week",
        "team",
        "full_name"
    ])
)

print(f"Records missing GSIS ID: {weekly_roster_missing_ids.height}")

weekly_roster_missing_ids.head(30)

Records missing GSIS ID: 121


season,week,team,full_name,football_name,position,depth_chart_position,status,espn_id,pfr_id,sportradar_id
i32,i32,str,str,str,str,str,str,str,str,str
2018,1,"""NE""","""Darren Andrews""","""Darren""","""WR""","""WR""","""RSN""",null,null,null
2018,2,"""NE""","""Darren Andrews""","""Darren""","""WR""","""WR""","""RSN""",null,null,null
2018,2,"""SF""","""Frank Stephens""","""Frank""","""WR""","""WR""","""DEV""",null,null,null
2018,3,"""NE""","""Darren Andrews""","""Darren""","""WR""","""WR""","""RSN""",null,null,null
2018,3,"""SF""","""Frank Stephens""","""Frank""","""WR""","""WR""","""DEV""",null,null,null
…,…,…,…,…,…,…,…,…,…,…
2018,14,"""SF""","""Frank Stephens""","""Frank""","""WR""","""WR""","""CUT""",null,null,null
2018,15,"""NE""","""Darren Andrews""","""Darren""","""WR""","""WR""","""RSN""",null,null,null
2018,15,"""NYG""","""Brock Miller""","""Brock""","""P""","""P""","""CUT""",null,null,null


In [73]:
weekly_rosters_clean = (
    weekly_rosters_reg
    .select([
        "season",
        "week",
        "team",
        "gsis_id",
        "full_name",
        "football_name",
        "position",
        "depth_chart_position",
        "status",
        "status_description_abbr",
        "years_exp",
        "entry_year",
        "rookie_year",
        "draft_club"
    ])
    .sort([
        "season",
        "week",
        "team",
        "gsis_id"
    ])
)

print(f"Clean weekly roster records: {weekly_rosters_clean.height:,}")
weekly_rosters_clean.head(10)

Clean weekly roster records: 475,749


season,week,team,gsis_id,full_name,football_name,position,depth_chart_position,status,status_description_abbr,years_exp,entry_year,rookie_year,draft_club
i32,i32,str,str,str,str,str,str,str,str,i32,i32,i32,str
2015,1,"""ARI""","""00-0019435""","""Mike Leach""","""Mike""","""LS""",null,"""ACT""","""A01""",15,2000,2000,null
2015,1,"""ARI""","""00-0021429""","""Carson Palmer""","""Carson""","""QB""",null,"""ACT""","""A01""",12,2003,2003,"""CIN"""
2015,1,"""ARI""","""00-0021998""","""Cory Redding""","""Cory""","""DE""",null,"""RES""","""A01""",12,2003,2003,"""DET"""
2015,1,"""ARI""","""00-0022921""","""Larry Fitzgerald""","""Larry""","""WR""",null,"""ACT""","""A01""",11,2004,2004,"""ARZ"""
2015,1,"""ARI""","""00-0024306""","""Frostee Rucker""","""Frostee""","""DT""",null,"""ACT""","""A01""",9,2006,2006,"""CIN"""
2015,1,"""ARI""","""00-0025430""","""Drew Stanton""","""Drew""","""QB""",null,"""ACT""","""A01""",8,2007,2007,"""DET"""
2015,1,"""ARI""","""00-0025433""","""LaMarr Woodley""","""LaMarr""","""LB""",null,"""RES""","""A01""",8,2007,2007,"""PIT"""
2015,1,"""ARI""","""00-0025682""","""Lyle Sendlein""","""Lyle""","""C""",null,"""ACT""","""A01""",8,2007,2007,null
2015,1,"""ARI""","""00-0026164""","""Chris Johnson""","""Chris""","""RB""",null,"""RSR""","""A01""",7,2008,2008,"""TEN"""


In [74]:
weekly_rosters_clean_output = (
    PROCESSED_DIR / "weekly_rosters_clean.parquet"
)

weekly_rosters_clean.write_parquet(
    weekly_rosters_clean_output
)

print(
    f"Saved {weekly_rosters_clean.height:,} "
    f"weekly roster records to:"
)

print(weekly_rosters_clean_output)

Saved 475,749 weekly roster records to:
c:\Users\efriedman\Desktop\NFL-Season-Projections\data\processed\weekly_rosters_clean.parquet
